In [ ]:
%pip install requests python-dotenv pandas gradio tabulate

In [ ]:
from dotenv import load_dotenv
import os
import requests
import json
import time

load_dotenv()

DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")
WORKSPACE_URL    = os.getenv("WORKSPACE_URL")
GENIE_SPACE_ID   = os.getenv("GENIE_SPACE_ID")

HEADERS = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}
BASE_URL = f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{GENIE_SPACE_ID}"

print(f"BASE_URL URL: {BASE_URL}")
print(f"Token set : {'Yes' if DATABRICKS_TOKEN else 'NO - check .env file'}")

In [ ]:
def genie_query(question: str, debug: bool = False) -> dict:
    """
    Sends a question to Databricks Genie and returns:
    - Natural language answer
    - Generated SQL
    - Query results

    Flow:
    1. Start conversation
    2. Poll until completed
    3. Extract SQL + text from attachments
    4. Execute SQL to fetch results
    """

    # ─────────────────────────────────────────────
    # STEP 1: Start conversation
    # ─────────────────────────────────────────────
    try:
        start_resp = requests.post(
            f"{BASE_URL}/start-conversation",
            headers=HEADERS,
            json={"content": question},
            timeout=30
        )
    except Exception as e:
        return {"error": f"Request failed: {str(e)}"}

    if start_resp.status_code != 200:
        return {"error": f"Failed to start conversation: {start_resp.text}"}

    start_data = start_resp.json()
    conversation_id = start_data["conversation_id"]
    message_id = start_data["message_id"]

    if debug:
        print(f"Conversation started: {conversation_id}")

    # ─────────────────────────────────────────────
    # STEP 2: Poll for completion
    # ─────────────────────────────────────────────
    max_wait = 60
    poll_interval = 2
    elapsed = 0

    message = None

    while elapsed < max_wait:
        poll_resp = requests.get(
            f"{BASE_URL}/conversations/{conversation_id}/messages/{message_id}",
            headers=HEADERS,
            timeout=30
        )

        if poll_resp.status_code != 200:
            return {"error": f"Polling failed: {poll_resp.text}"}

        message = poll_resp.json()
        status = message.get("status", "")

        if debug:
            print(f"Status: {status}")

        if status == "COMPLETED":
            break
        elif status in ("FAILED", "CANCELLED"):
            return {"error": f"Genie query {status}", "raw": message}

        time.sleep(poll_interval)
        elapsed += poll_interval

    if elapsed >= max_wait:
        return {"error": "Genie query timed out"}

    # ─────────────────────────────────────────────
    # STEP 3: Parse attachments
    # ─────────────────────────────────────────────
    attachments = message.get("attachments", [])

    sql_query = None
    description = None
    text_answer = ""

    for a in attachments:
        # Extract SQL
        if "query" in a:
            sql_query = a["query"].get("query")
            description = a["query"].get("description")

        # Extract natural language answer
        if "text" in a:
            text_answer += a["text"].get("content", "") + " "

    text_answer = text_answer.strip()

    if debug:
        print("\n--- Parsed Output ---")
        print("SQL:", sql_query)
        print("Text:", text_answer)

    # If no SQL → return text only
    if not sql_query:
        return {
            "answer": text_answer or "No answer returned by Genie.",
            "sql": None,
            "results": [],
            "num_rows": 0
        }

    # ─────────────────────────────────────────────
    # STEP 4: Execute SQL
    # ─────────────────────────────────────────────
    exec_resp = requests.get(
        f"{BASE_URL}/conversations/{conversation_id}/messages/{message_id}/query-result",
        headers=HEADERS,
        timeout=60
    )

    if exec_resp.status_code != 200:
        return {"error": f"Failed to fetch query results: {exec_resp.text}"}

    exec_data = exec_resp.json()

    # Extract schema
    columns = [
        col["name"]
        for col in exec_data.get("statement_response", {})
                            .get("manifest", {})
                            .get("schema", {})
                            .get("columns", [])
    ]

    # Extract rows
    rows = exec_data.get("statement_response", {}) \
                    .get("result", {}) \
                    .get("data_typed_array", [])

    results = []
    for row in rows:
        values = [v.get("str", None) for v in row.get("values", [])]
        results.append(dict(zip(columns, values)))

    # ─────────────────────────────────────────────
    # FINAL RESPONSE
    # ─────────────────────────────────────────────
    return {
        "answer": text_answer or description or "Query executed successfully.",
        "sql": sql_query,
        "results": results,
        "num_rows": len(results),
        "conversation_id": conversation_id
    }

# Test to verify connection

In [ ]:
# ============================================================
# Quick test — run this first to verify connection
# ============================================================
result = genie_query("Count hospitals in Accra with cardiology", debug=True)

if "error" in result:
    print("❌ Error:", result["error"])
    if "raw" in result:
        print("Raw response:", result["raw"])
else:
    print("✅ Answer:", result["answer"])
    print("SQL:", result["sql"])
    print("Rows:", result["num_rows"])


# Gradio UI

In [ ]:
import gradio as gr
import pandas as pd

def format_response(result: dict) -> str:
    if "error" in result:
        return f"Error: {result['error']}"

    answer  = result.get("answer", "")
    sql     = result.get("sql", "")
    results = result.get("results", [])
    rows    = result.get("num_rows", 0)

    formatted = f"{answer}\n\n"
    if sql:
        formatted += f"**Generated SQL:**\n```sql\n{sql}\n```\n\n"

    if rows > 0 and results:
        df = pd.DataFrame(results)
        # Convert dataframe to markdown
        formatted += f"**Results ({rows} rows):**\n"
        formatted += df.head(10).to_markdown(index=False)
        if rows > 10:
            formatted += "\n\n*(showing first 10 rows)*"

    return formatted

def respond(message, history):
    result = genie_query(message)
    return format_response(result)

with gr.Blocks(title="Genie Text-to-SQL Assistant") as demo:
    gr.Markdown("## Genie Text-to-SQL Assistant")
    gr.Markdown("Ask questions to query your data directly using plain English. Powered by Databricks Genie.")

    gr.ChatInterface(
        fn       = respond,
        examples = [
            ["What are the top 5 facilities?"],
            ["Count facilities by region"],
            ["Show me hospitals in Greater Accra"]
        ],
        title = "",
    )

demo.launch()
